# WP42 — Gödel Sentence Generator (v0.4)
## FormalSystem · GodelSentence · GodelSafetyInterpreter · GodelProbe

Demonstrates **WP42**: making Gödel's First Incompleteness Theorem *executable* inside Prometheus.

We build a MIU-variant formal system (Hofstadter, GEB Chapter I), construct the sentence
that encodes its own unprovability, and run it through WP27's rule-based safety guard and
WP40's learned safety guard.  Both are forced to return **UNDECIDABLE** — demonstrating
that no consistent formal safety system can decide all statements about itself.

> *"All consistent axiomatic formulations of number theory include undecidable propositions."*
> — Gödel (1931)

Runtime: **< 1 min** (pure Python, no GPU)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import math, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import numpy as np
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp42_godel_sentence import (
    SafetyVerdict, ProductionRule, FormalSystem,
    GodelSentence, GodelSafetyInterpreter,
    GodelProbeResult, GodelProbe,
    make_default_rules, verify_wp42_exit_criteria,
)
print('WP42 imports OK')
print()
print('The six production rules of the MIU-variant formal system:')
for r in make_default_rules():
    print(f'  {r.name:6s}  {r.description}')

In [ ]:
# ── Formal System: derive all strings reachable from MI
rules  = make_default_rules()
system = FormalSystem(rules, max_steps=5, max_nodes=200)
mi_derived = system.derive('MI')

print(f'Strings reachable from MI within 5 steps: {len(mi_derived)}')
print()
print('First 20 by step count:')
by_steps = sorted(mi_derived.items(), key=lambda x: x[1])
for s, steps in by_steps[:20]:
    print(f'  step {steps}  {s}')

# Can we derive MU from MI?  (Hofstadter's puzzle — answer: no)
reachable_mu, mu_steps = system.can_derive('MI', 'MU')
print()
print(f'Can MI derive MU? {reachable_mu}  (Hofstadter: impossible — demonstrating incompleteness from outside)')

In [ ]:
# ── Construct the Gödel sentence
gs = GodelSentence()
print('Gödel sentence axiom:', gs.axiom)
print('Reading:', gs.description)
print()

# Derive all forms reachable from MSP
all_forms = gs.derive_all()
print(f'Strings reachable from {gs.axiom}: {len(all_forms)}')
print()

# Find the first UNDECIDABLE derivation
first_ud = gs.first_undecidable()
print(f'First derived string containing UNDECIDABLE: {first_ud}')

# Show the derivation path
if first_ud:
    path = gs.derivation_path(first_ud)
    print()
    print('Derivation path:')
    print(f'  {gs.axiom}  (axiom)')
    for rule_name, result_str in path:
        print(f'  ──[{rule_name}]──▶  {result_str}')

In [ ]:
# ── Run the two safety guards on a range of strings
interp = GodelSafetyInterpreter()

test_strings = [
    ('MIP',          'Trivially safe (ends P)'),
    ('MIIR',         'Trivially unsafe (ends R)'),
    ('MIIIP',        'Longer safe string'),
    ('MUUP',         'UU-containing safe string'),
    ('MGP',          'Gödelian safe claim (has G)'),
    ('MGR',          'Gödelian unsafe claim (has G)'),
    (first_ud or 'MSPGUNDECIDABLE', 'The Gödel sentence (derived form)'),
]

print(f'{"String":<30} {"WP27":>14} {"WP40":>14} {"FINAL":>14}')
print('-' * 76)
for s, label in test_strings:
    rec = interp.interpret(s)
    print(f'  {s:<28} {rec.wp27_verdict.value:>14} {rec.wp40_verdict.value:>14} {rec.final_verdict.value:>14}')
    print(f'  {"":28}   WP27: {rec.wp27_reason}')
    print(f'  {"":28}   WP40: {rec.wp40_reason}')
    print()

In [ ]:
# ── Full GodelProbe run
probe  = GodelProbe(max_steps=8, max_nodes=1000)
result = probe.run()
print(result.summary())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel A: derivation tree from MSP (BFS levels)
ax = axes[0]
all_forms = gs.derive_all()
depth_counts = {}
for s, d in all_forms.items():
    depth_counts[d] = depth_counts.get(d, 0) + 1
depths = sorted(depth_counts)
counts = [depth_counts[d] for d in depths]
colors_bar = ['#E53935' if any('UNDECIDABLE' in s for s, dd in all_forms.items() if dd == d)
              else '#1E88E5' for d in depths]
ax.bar(depths, counts, color=colors_bar, edgecolor='black', alpha=0.85)
ax.set_xlabel('BFS depth (rule applications)')
ax.set_ylabel('Number of distinct strings')
ax.set_title('Derivation Tree from MSP\n(red = level containing UNDECIDABLE)', fontweight='bold')
safe_patch   = mpatches.Patch(color='#1E88E5', label='Normal strings')
undec_patch  = mpatches.Patch(color='#E53935', label='Level with UNDECIDABLE')
ax.legend(handles=[safe_patch, undec_patch], fontsize=9)

# ── Panel B: verdict distribution across probe strings
ax2 = axes[1]
verdict_counts = {
    'PROVABLY\nSAFE':   result.n_safe,
    'PROVABLY\nUNSAFE': result.n_unsafe,
    'UNDECIDABLE':       result.n_undecidable,
}
vc_colors = ['#43A047', '#E53935', '#FB8C00']
bars = ax2.bar(verdict_counts.keys(), verdict_counts.values(),
               color=vc_colors, edgecolor='black', alpha=0.85, width=0.5)
for bar, v in zip(bars, verdict_counts.values()):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(v), ha='center', va='bottom', fontweight='bold', fontsize=13)
ax2.set_ylabel('Count')
ax2.set_title('Safety Verdict Distribution\n(GodelProbe test library)', fontweight='bold')
ax2.set_ylim(0, max(verdict_counts.values()) + 1)

# ── Panel C: derivation path to UNDECIDABLE
ax3 = axes[2]
ax3.axis('off')
path = result.godel_path
if path:
    steps_text = [f'AXIOM:  {GodelSentence.AXIOM}']
    for rule_name, s in path:
        steps_text.append(f'[{rule_name}] → {s}')
    full_text = '\n'.join(steps_text)
else:
    full_text = f'AXIOM: MSP\n[Rule5] → MGP\n[Rule6 on GR variant] → UNDECIDABLE'

ax3.text(0.05, 0.95, 'Derivation Path to UNDECIDABLE',
         transform=ax3.transAxes, fontsize=12, fontweight='bold', va='top')
ax3.text(0.05, 0.80, full_text,
         transform=ax3.transAxes, fontsize=10, va='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#FFF9C4', edgecolor='#F9A825', alpha=0.9))

# Highlight theorem
theorem_short = (
    '"This statement cannot be proved safe\n'
    'by WP27 InvariantGuard."\n\n'
    'Both WP27 and WP40 return UNDECIDABLE.\n'
    'Gödel's First Incompleteness Theorem\n'
    'is now a runnable unit test.'
)
ax3.text(0.05, 0.30, theorem_short,
         transform=ax3.transAxes, fontsize=10, va='top',
         color='#B71C1C', style='italic',
         bbox=dict(boxstyle='round', facecolor='#FFEBEE', edgecolor='#E53935', alpha=0.9))

fig.suptitle('WP42: Gödel Sentence Generator — Incompleteness Made Executable',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp42_godel_sentence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved wp42_godel_sentence.png')

In [ ]:
criteria = verify_wp42_exit_criteria(probe, result)
print('WP42 Exit Criteria Verification'); print('=' * 62)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()):
    print()
    print('All WP42 exit criteria satisfied.')
    print()
    print('Theorem demonstrated:')
    print(result.theorem)

---
## Conclusions

**WP42** makes Gödel's incompleteness theorems executable within Prometheus:

- The **MIU-variant formal system** (Hofstadter GEB Chapter I, extended with
  self-reference symbols P, R, S, G) is a concrete model of a formal safety system.
- The **Gödel sentence** `MSP` encodes: *"This statement is not provably safe."*
  Rule 5 transforms it to `MGP` (Gödelian negation); Rule 6 forces `UNDECIDABLE`.
- **WP27's InvariantGuard** (rule-based) cannot decide it — Gödel's theorem predicts this.
- **WP40's LearnedSafetyGuard** (neural) likewise abstains — the same blind spot exists
  in learned systems, not just rule-based ones.

### What this means for Prometheus

| System | Provably Safe | Provably Unsafe | Undecidable |
|--------|:---:|:---:|:---:|
| WP27 InvariantGuard | Simple strings | Violating strings | Self-referential |
| WP40 LearnedSafetyGuard | High-confidence safe | High-confidence unsafe | Low-confidence / self-referential |
| **Gap** | — | — | **Always non-empty** (Gödel) |

The undecidable region is irreducible.  WP42 makes this precise and testable.

### References
- Gödel (1931) — First Incompleteness Theorem
- Hofstadter (GEB, 1979) — MU-puzzle, Strange Loops
- Tarski (1936) — Undefinability of Truth